In [1]:
import torch
import numpy as np
from src.data.sequence import Sequence  # 替换为你的实际模块路径

# 构造示例数据
t_start = 0.0
arrival_times = np.array([1.0, 2.5, 4.0, 6.0])  # 事件发生时间
t_end = 8.0

# 计算 inter_times: Δt_0, Δt_1, ..., Δt_N, Δt_survival
inter_times = np.diff(arrival_times, prepend=[t_start], append=[t_end])  # shape = [5]

# 额外属性，如震级 (mag) 和位置 (loc)
mag = np.array([3.1, 2.8, 4.0, 3.5])  # shape = [4]
loc = np.array([
    [30.5, 102.3],
    [30.6, 102.4],
    [30.7, 102.5],
    [30.8, 102.6],
])  # shape = [4, 2]

# 创建 Sequence 对象
seq = Sequence(
    inter_times=inter_times,
    t_start=t_start,
    mag=mag,
    loc=loc
)

# ✅ 查看完整事件序列
print("Original arrival times:", seq.arrival_times)
print("Original inter_times:", seq.inter_times)
print("Original magnitude:", seq['mag'])

# ✅ 调用 get_subsequence（例如截取 [2.0, 5.0] 之间的事件）
sub_seq = seq.get_subsequence(start=2.0, end=5.0)

# ✅ 输出子序列内容
print("\n--- Subsequence ---")
print("Arrival times:", sub_seq.arrival_times)
print("Inter times:", sub_seq.inter_times)
print("Magnitude:", sub_seq['mag'])
print("Location:", sub_seq['loc'])


Original arrival times: tensor([1.0000, 2.5000, 4.0000, 6.0000], dtype=torch.float64)
Original inter_times: tensor([1.0000, 1.5000, 1.5000, 2.0000, 2.0000], dtype=torch.float64)
Original magnitude: tensor([3.1000, 2.8000, 4.0000, 3.5000], dtype=torch.float64)

--- Subsequence ---
Arrival times: tensor([2.5000, 4.0000], dtype=torch.float64)
Inter times: tensor([0.5000, 1.5000, 1.0000], dtype=torch.float64)
Magnitude: tensor([2.8000, 4.0000], dtype=torch.float64)
Location: tensor([[ 30.6000, 102.4000],
        [ 30.7000, 102.5000]], dtype=torch.float64)


In [8]:
from src.data.batch import Batch 
buffer_batch = Batch.init_sample_batch(past_seq=sub_seq, batch_size=2, max_sample_len=5)

In [9]:
print(buffer_batch)

Batch(
  inter_times: [2, 8],
  arrival_times: [2, 8],
  t_start: [2],
  t_end: [2],
  t_nll_start: [2],
  nll_event_mask: [2, 8],
  input_mask: [2, 8],
  start_idx: [2],
  end_idx: [2],
  non_pad_mask: [2, 8],
  type_seq: [2, 8],
  mag: [2, 8],
  loc: [2, 8, 2]
)


In [10]:
buffer_batch.get_tmp_batch()

Batch(
  inter_times: [2, 1],
  arrival_times: [2, 1],
  end_idx: [2],
  input_mask: [2, 1],
  nll_event_mask: [2, 1],
  non_pad_mask: [2, 1],
  type_seq: [2, 1],
  mag: [2, 1],
  loc: [2, 1, 2]
)

In [2]:
sub_event_seq = sub_seq.to_event_sequence()

In [3]:
print(sub_event_seq.arrival_times,sub_event_seq.inter_times)

tensor([0.0000, 1.5000], dtype=torch.float64) tensor([0.0000, 1.5000], dtype=torch.float64)


In [4]:
print(sub_seq.arrival_times)

tensor([2.5000, 4.0000], dtype=torch.float64)


In [5]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset([seq])

In [6]:
dl = ds.get_dataloader(batch_size=2, shuffle=True)

In [7]:
for batch in dl:
    print("\n--- Batch ---")
    print("Arrival times:", batch['arrival_times'])
    print("Inter times:", batch['inter_times'])
    print("Magnitude:", batch['mag'])
    print("Location:", batch['loc'])
    break  # 只打印第一个批次


--- Batch ---
Arrival times: tensor([[1.0000, 2.5000, 4.0000, 6.0000, 8.0000]], dtype=torch.float64)
Inter times: tensor([[1.0000, 1.5000, 1.5000, 2.0000, 2.0000]], dtype=torch.float64)
Magnitude: tensor([[3.1000, 2.8000, 4.0000, 3.5000, 0.0000]], dtype=torch.float64)
Location: tensor([[[ 30.5000, 102.3000],
         [ 30.6000, 102.4000],
         [ 30.7000, 102.5000],
         [ 30.8000, 102.6000],
         [  0.0000,   0.0000]]], dtype=torch.float64)


In [8]:
event_ds = TppDataset([sub_event_seq]) 
event_dl = event_ds.get_dataloader(batch_size=2, shuffle=True)
for batch in event_dl:
    print("\n--- Event Batch ---")
    print("Arrival times:", batch['arrival_times'])
    print("Inter times:", batch['inter_times'])
    print("Magnitude:", batch['mag'])
    print("Location:", batch['loc'])
    print("Type seq:", batch['type_seq'])
    break  # 只打印第一个批次


--- Event Batch ---
Arrival times: tensor([[0.0000, 1.5000]])
Inter times: tensor([[0.0000, 1.5000]])
Magnitude: tensor([[2.8000, 4.0000]], dtype=torch.float64)
Location: tensor([[[ 30.6000, 102.4000],
         [ 30.7000, 102.5000]]], dtype=torch.float64)
Type seq: tensor([[0, 0]])


In [4]:
import torch

def get_time_window_mask(timestamps: torch.Tensor, window_size: float) -> torch.Tensor:
    """
    Sliding-window causal mask  (True=被掩蔽, False=可见)
    """
    diff = timestamps.unsqueeze(2) - timestamps.unsqueeze(1)   
    mask = (diff < 0) | (diff > window_size)            
    return mask.bool()

# 两条序列各 6 个时间戳
t = torch.tensor([[0., 1., 3., 8., 12., 13.],
                  [0., 2., 4., 7., 11., 20.]])

mask = get_time_window_mask(t, window_size=5.)

print("序列 0 的掩码 (0=可见, 1=被掩蔽):")
print(mask[0].int())
print("\n序列 1 的掩码:")
print(mask[1].int())


序列 0 的掩码 (0=可见, 1=被掩蔽):
tensor([[0, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [1, 1, 0, 0, 1, 1],
        [1, 1, 1, 0, 0, 1],
        [1, 1, 1, 0, 0, 0]], dtype=torch.int32)

序列 1 的掩码:
tensor([[0, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [1, 0, 0, 0, 1, 1],
        [1, 1, 1, 0, 0, 1],
        [1, 1, 1, 1, 1, 0]], dtype=torch.int32)


In [2]:
def get_subsequent_mask(seq):
    """ For masking out the subsequent info, i.e., masked self-attention. """
    assert seq.dim() == 2
    sz_b, len_s = seq.size()
    subsequent_mask = torch.triu(
        torch.ones((len_s, len_s), device=seq.device, dtype=torch.uint8), diagonal=1)
    subsequent_mask = subsequent_mask.unsqueeze(0).expand(sz_b, -1, -1)  # b x ls x ls
    return subsequent_mask

get_subsequent_mask(t)

tensor([[[0, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1],
         [0, 0, 0, 0, 1, 1],
         [0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0]],

        [[0, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1],
         [0, 0, 0, 0, 1, 1],
         [0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0]]], dtype=torch.uint8)